# Zadaća 10

Proučite [House Price Regression Dataset](https://www.kaggle.com/datasets/prokshitha/home-value-insights) i definirajte što je target varijabla. Iz dataseta izbacite sve značajke koje nisu numeričke osim jedne, a ako negdje nedostaje vrijednost umetnite medijan. Tu jednu kategorijsku značajku konvertirajte u numeričke korištenjem OneHotEncoder-a ili slične metode. Model trenirajte sa linearnom regresijom. Dataset pripremite sa train_test_split funkcijom i pri tom koristite random_state=21, a 20% neka vam ostane za testiranje. Izračunajte RMSE sa testnim podacima. Ispišite izračunato.

In [198]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd

In [199]:
df = pd.read_csv('house_price_regression_dataset.csv', low_memory=False)
df.head()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06


U dobivenom datasetu ne postoje kategoričke varijable, te sukladno dogovoru uzimam drugi dataset [CO<sub>2</sub> emission of cars](https://www.kaggle.com/datasets/midhundasl/co2-emission-of-cars-dataset).

In [200]:
df = pd.read_csv('co2-emission-of-cars-dataset.csv', low_memory=False)
df.head()

,Car,Model,Volume,Weight,CO2,Unnamed: 5
0,Toyota,Aygo,1000,790,99,NaN
1,Mitsubishi,Space Star,1200,1160,95,NaN
2,Skoda,Citigo,1000,929,95,NaN
3,Fiat,500,900,865,90,NaN
4,Mini,Cooper,1500,1140,105,NaN


U datasetup postoji pogreška sa dodatnim zarezom na kraju svakog retka što rezultira dodatnim stupcem kojeg ću obrisati.

In [201]:
df = df.drop(columns=['Unnamed: 5'])
df.head()

,Car,Model,Volume,Weight,CO2
0,Toyota,Aygo,1000,790,99
1,Mitsubishi,Space Star,1200,1160,95
2,Skoda,Citigo,1000,929,95
3,Fiat,500,900,865,90
4,Mini,Cooper,1500,1140,105


In [202]:
df.isna().sum()

Car       0
Model     0
Volume    0
Weight    0
CO2       0
dtype: int64

U datasetu ne postoje vrijednosti koje nedostaju.

In [203]:
df.value_counts()

Car         Model       Volume  Weight  CO2
Audi        A1          1600    1150    99     1
            A4          2000    1490    104    1
Mercedes    SLK         2500    1395    120    1
Mini        Cooper      1500    1140    105    1
Mitsubishi  Space Star  1200    1160    95     1
Opel        Astra       1600    1330    97     1
            Insignia    2000    1428    99     1
            Zafira      1600    1405    109    1
Skoda       Citigo      1000    929     95     1
            Fabia       1400    1109    90     1
            Octavia     1600    1415    99     1
            Rapid       1600    1119    104    1
Suzuki      Swift       1300    990     101    1
Toyota      Aygo        1000    790     99     1
VW          Up!         1000    929     105    1
Volvo       S60         2000    1415    99     1
            V70         1600    1523    109    1
Mercedes    E-Class     2100    1605    115    1
            CLA         1500    1465    102    1
            C-Class     2

Mičem 'Car' značajku kako bi ostala samo jedna kategorička varijabla.

In [204]:
df = df.drop(columns=['Car'])

In [205]:
numerical_columns=['Volume', 'Weight']
categorical_columns = ['Model']

X1 = df[numerical_columns]
y1 = df['CO2']

In [206]:
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=21)
algorithm1 = LinearRegression()
model1 = algorithm1.fit(X1_train, y1_train)
predictions1 = model1.predict(X1_test)

In [207]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(drop=None, handle_unknown='ignore'), categorical_columns),
        ('numerical', "passthrough", numerical_columns)
    ],
    remainder='drop'
)

algorithm2 = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('linear', LinearRegression())
    ]
)

X2 = df[numerical_columns + categorical_columns]
y2 = df['CO2']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=21)
model2 = algorithm2.fit(X2_train, y2_train)
predictions2 = algorithm2.predict(X2_test)

In [208]:
print('MSE 1: ', mean_squared_error(y1_test, predictions1))
print('R^2 1:', model1.score(X1_test, y1_test))
print('MSE 2: ', mean_squared_error(y2_test, predictions2))
print('R^2 2:', algorithm2.score(X2_test, y2_test))

MSE 1:  33.62698916004156
R^2 1: -0.31951398298139777
MSE 2:  46.54836921053804
R^2 2: -0.8265454503215417
